# Notebook to preprocess IFRC reports

## Steps
1. Load reports text from JSON (check it is the correct version, eventually redo scraping ourself)
2. Filter out unnessecary reports
3. Clean text
4. Separate sentences and tokenize
5. Add hazard category for each report (use Laura's reclassifying)
6. Add division according to header


In [296]:
import pandas as pd
import json
from collections import Counter
from text_processing_functions import *
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
import re

In [297]:
# DATA_FOLDER = '../Data_backup'
DATA_FOLDER = '/scratchx/lhasbini/como_school'

In [298]:
# file_path = DATA_FOLDER + '/all_ifrc_reports_info_unnested.json' #not sure if this is the correct file

# # Open and read the JSON file
# with open(file_path, 'r') as json_file:
#     all_ifrc_reports_info_unnested = json.load(json_file)

In [299]:
file_path = DATA_FOLDER + '/filtered_report_types_nat_hazards.json' #not sure if this is the correct file

# Open and read the JSON file
with open(file_path, 'r') as json_file:
    all_ifrc_reports_info_unnested = json.load(json_file)

In [300]:
# Convert the JSON data into a Pandas DataFrame
data = pd.DataFrame(all_ifrc_reports_info_unnested)

In [301]:
data

,reportName,disasterType,date,reportLink,location,appealCode,appealType,origType,pdfDownloaded,text,disasterTypeReclassified,disasterTypeFlag,naturalHazard,text_processed,sentences
0,Boliva - Wildfires (MDRBO017),Fire,23/09/2024,https://adore.ifrc.org/Download.aspx?FileId=83...,Bolivia,MDRBO017,DREF Operation,MDRBO017do,1,DREF Operation\nBolivia: Wildfires\nFirst aid ...,Wildfire,0,1,DREF Operation Bolivia: Wildfires First aid at...,[DREF Operation Bolivia: Wildfires First aid a...
1,Algeria - Flood (MDRDZ011),Famine / Food Insecurity,22/09/2024,https://adore.ifrc.org/Download.aspx?FileId=83...,Algeria,MDRDZ011,DREF Operation,MDRDZ011do,1,DREF Operation\nAlgeria Flood 2024 Bechar\nApp...,Flood,0,1,DREF Operation Algeria Flood 2024 Bechar Appea...,[DREF Operation Algeria Flood 2024 Bechar Appe...
2,Pakistan - Floods (MDRPK026),Flood,17/09/2024,https://adore.ifrc.org/Download.aspx?FileId=83...,Pakistan,MDRPK026,DREF Operation,MDRPK026do,1,DREF Operation\nPakistan Flood August 2024\nRe...,Flood,0,1,DREF Operation Pakistan Flood August 2024 Resi...,[DREF Operation Pakistan Flood August 2024 Res...
3,Cameroon - Floods (MDRCM039),Flood,13/09/2024,https://adore.ifrc.org/Download.aspx?FileId=83...,Cameroon,MDRCM039,DREF Operation,MDRCM039do,1,DREF Operation\nCameroon_Far North Floods\nEva...,Flood,0,1,DREF Operation Cameroon_Far North Floods Evacu...,[DREF Operation Cameroon_Far North Floods Evac...
4,Benin - Floods (MDRBJ019),Flood,07/09/2024,https://adore.ifrc.org/Download.aspx?FileId=83...,Benin,MDRBJ019,DREF Operation,MDRBJ019do,1,DREF Operation\nBenin_Flood in Lalo\nField vis...,Flood,0,1,DREF Operation Benin_Flood in Lalo Field visit...,[DREF Operation Benin_Flood in Lalo Field visi...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1856,Serbia - Flash Floods (MDRRS009),Pluvial/Flash Flood,27/03/2015,https://adore.ifrc.org/Download.aspx?FileId=76078,Serbia,MDRRS009,Operations Update,Operations Update No. 3,1,\nEmergency appeal n° MDRRS009 \nGLIDE n° FF-...,Flood,0,1,Emergency appeal n° MDRRS009 GLIDE n° FF-2014-...,[Emergency appeal n° MDRRS009 GLIDE n° FF-2014...
1857,Chile - Fire Valparaiso (MDRCL010),Fire,24/03/2015,https://adore.ifrc.org/Download.aspx?FileId=74617,Chile,MDRCL010,DREF Operation Final Report,DREF Final Report,1,P a g e . | 1 \n \n \nDREF Operation \nOperat...,Wildfire,0,1,P a g e . | 1 DREF Operation Operation no. MDR...,"[P a g e ., | 1 DREF Operation Operation no., ..."
1858,Argentina - Floods (MDRAR008),Flood,24/03/2015,https://adore.ifrc.org/Download.aspx?FileId=74605,Argentina,MDRAR008,DREF Operation,DREF Operation,1,P a g e | 1 \n \n \nDREF Operation \nMDRAR0...,Flood,0,1,P a g e | 1 DREF Operation MDRAR008 Glide no. ...,[P a g e | 1 DREF Operation MDRAR008 Glide no....
1859,Costa Rica - Volcanic Eruption Resp. Prep. (MD...,Volcanic Eruption,23/03/2015,https://adore.ifrc.org/Download.aspx?FileId=74470,Costa Rica,MDRCR012,DREF Operation,DREF operation,1,P a g e | 1 \n \n \n \nAsh c\nloud a\nbove th...,Volcano,0,1,P a g e | 1 Ash c loud a bove the Turrialba vo...,[P a g e | 1 Ash c loud a bove the Turrialba v...


## Filtering useless reports

In [5]:
## filter out useless reports
filtered_reports = [
    disaster_report for disaster_report in all_ifrc_reports_info_unnested
    if disaster_report['appealType'] in ['Operations Update', 'DREF Operation', 'DREF Operation Final Report', 'DREF Operation Update']
]

## Text preprocessing

In [6]:
#load libraries fo nlp
#not clear exactly which preprocessing steps must be undertaken
import nltk
from nltk.tokenize import sent_tokenize
nltk.download('punkt_tab')
nltk.download('punkt')  # Download sentence tokenizer
nltk.download('stopwords') # Download stopwords

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /Users/lseverino/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [7]:
#clean text and tokenize into sentences
for item in filtered_reports[:]:
    if 'text' in item:
        item['text_processed'] = clean_text(item['text'])
        item['sentences'] = sent_tokenize(item['text_processed'])
    else: # drop reports without text
        filtered_reports.remove(item)

In [8]:
# Add the ISO code to each dict in the list

import pycountry

for report in filtered_reports[:]:
    country_name = report.get("location")
    try:
        # Lookup the ISO code using pycountry
        country = pycountry.countries.get(name=country_name)
        if country:
            report["iso_code"] = country.alpha_3  # Adds the ISO 3166-1 Alpha-3 code
        else:
            report["iso_code"] = "Unknown"
    except KeyError:
        report["iso_code"] = "Unknown"


## Add natural hazard type and filter out other disasters

In [11]:
#load emdat data
emdat_data = pd.read_excel('../Data_backup/public_emdat_incl_hist_2024-09-09.xlsx')

In [303]:
disgroup = ['Natural']
dissubgroup = ['Meteorological', 'Hydrological', 'Climatological']
emdat_data_nathaz = emdat_data.where(emdat_data['Disaster Group'].isin(disgroup)).dropna(how='all') #maybe also keep earthquakes etc.
haz_types_emdat = list(emdat_data_nathaz['Disaster Subtype'].unique()) + list(emdat_data_nathaz['Disaster Type'].unique())
haz_types_emdat = [re.sub(r' \(.*?\)', '', disaster) for disaster in haz_types_emdat]

In [304]:
# haz_types_emdat

In [305]:
### here need dict to identify types of hazards according to emdata
hazard_patterns = {
        'Drought': r"drought.*|dry spell.*",
        'Flood': r"\b(flood|floods|flooding|inundation|inundations|glacial lake outburst)\b",
        'Storm': r"storm.*|superstorm.*|tornado.*|windstorm.*|snowstorm.*|snowfal.*|blizzard.*|derecho.*|winterstorm.*|hail.*|extra tropical storm.*|thunderstorm.*",
        #'Tornado': r"tornado.*",
        #'Hurricane': r"hurricane.*",
        'Storm surge': r"storm surge.*",
        'Heat wave': r"heat wave.*|heatwave.*|heat episode.*|((heat|hot) spell).*|heat stress.*",
        'Cold wave': r"cold wave.*|coldwave.*|severe winter conditions.*|cold spell.*",
        'Mass movement': r"land slide.*|landslide.*|rockfall.*|mudslide.*|mass movement.*",
        #'Earthquake': r"earthquake.*",
        'Tropical cyclone': r"cyclone.*|tropical cyclone.*|hurricane.*|typhoon.*",
        #'Volcano': r"volcan.*",
        'Tidal Wave': r"tidal wav.*",
        'Wildfire': r"fire.*|forestfire.*|wildfire.*|landfire.*|bushfire.*|forest fire.*|wild fire.*|land fire.*|bush fire.*"
    }

In [306]:
# Apply the function to the 'text_preprocessed' column of the DataFrame
for report in filtered_reports[:]:
    if 'text' in report:
        if 'text_processed' not in report:
            report['text_processed'] = clean_text(report['text'])#for djibouti
        report['hazards_found'] = check_hazard_type_keyword(report['text_processed'], hazard_patterns)
    else:
        print(report)#why is djibouti removed
        filtered_reports.remove(report)

In [307]:
# filter out reports not corresponding to emdat nathaz
for report in filtered_reports[:]:
    any_overlap = False
    if len(report['hazards_found']) > 0:
        any_overlap = np.any([hazard in emdat_data_nathaz for hazard in report['hazards_found']])
        if not any_overlap:
            filtered_reports.remove(report)
    else:
        filtered_reports.remove(report)

## Select subsections containing natural hazard info

In [308]:
for report in filtered_reports[:]:
    report['nathaz_text'] = select_hazard_description(report['sentences'])

## Save data

In [27]:
# file_path = DATA_FOLDER + '/filtered_report_types_nat_hazards_summary-header.json' #not sure if this is the correct file
# with open(file_path, 'w') as f:
#     json.dump(filtered_reports, f, indent=4)